### ЗАДАЧА: Бронирование переговорок

Офис-менеджер получает пачку заявок на переговорки.
Нужно собрать систему, которая:
- принимает корректные бронирования,
- отбрасывает конфликтующие или неправильные заявки,
- хранит расписание по комнатам,
- позволяет быстро понять, какая переговорка загружена сильнее всего.

В некоторых заявках указана неизвестная комната,
в некоторых время перепутано,
а некоторые пересекаются с уже занятыми слотами.


In [ ]:
from dataclasses import dataclass
from typing import Optional


rooms = {'A-101', 'B-204', 'C-305'}
# rows: booking_id|room_id|owner|start_hour|end_hour
rows = [
    'BK-100|A-101|Alice|9|11',
    'BK-101|A-101|Bob|10|12',
    'BK-102|B-204|Kira|13|15',
    'BK-103|X-999|Oleg|11|12',
    'BK-104|C-305|Eva|16|15',
    'BK-105|B-204|Max|15|17',
]


class BookingError(Exception):
    pass


class RoomNotFoundError(BookingError):
    pass


class TimeRangeError(BookingError):
    pass


class TimeConflictError(BookingError):
    pass


@dataclass(order=True)
class Booking:
    start_hour: int
    end_hour: int
    booking_id: str
    room_id: str
    owner: str


class RoomSchedule:
    def __init__(self, room_id):
        self.room_id = room_id
        # TODO: сохранить room_id

        self.bookings: list[Booking] = []
        # TODO: создать список bookings

    def can_add(self, booking: Booking) -> bool:
        # TODO: пройтись по уже существующим booking в self.bookings
        for existing_booking in self.bookings:
            if not (booking.end_hour <= existing_booking.start_hour or booking.start_hour >= existing_booking.end_hour):
                return False
            return True
        # TODO: проверить пересечение интервалов
        # TODO: если пересечение есть -> вернуть False
        # TODO: если конфликтов нет -> вернуть True


    def add_booking(self, booking: Booking):
        if not self.can_add(booking):
            raise TimeConflictError("Ошибка бронирования")
        # TODO: если can_add(...) == False -> raise TimeConflictError(...)

        # TODO: добавить booking в self.bookings
        self.bookings.append(booking)

        # TODO: отсортировать self.bookings
        self.bookings.sort(key = lambda x: x.start_hour)

    def total_reserved_hours(self) -> int:
        # TODO: вернуть сумму (end_hour - start_hour) по всем бронированиям комнаты
        total = 0
        for booking in self.bookings:
            total += booking.end_hour - booking.start_hour
        return total
        

class BookingService:
    def __init__(self, rooms):
        # TODO: создать schedules вида room_id -> RoomSchedule(room_id)
        self.shedules: dict[str, RoomSchedule] = {}

        # TODO: создать списки accepted и errors
        self.accepted: list[Booking] = []
        self.errors: list[tuple[str]] = []

    def parse_booking(self, row: str) -> Booking:
        # TODO: split по '|'
        parts = row.split("|")
        if len(parts) != 5:
            raise BookingError("Неверный формат")
        
        # TODO: ожидать 5 частей: booking_id, room_id, owner, start_raw, end_raw
        booking_id, room_id, owner, start_raw, end_raw = parts

        # TODO: start_raw и end_raw преобразовать в int
        try:
            start_hour = int(start_raw)
            end_hour = int(end_raw) 
        except ValueError as e: 
            raise BookingError ("Неправильный формат времени")

        # TODO: если room_id не существует -> RoomNotFoundError
        if room_id not in self.shedules:
            raise RoomNotFoundError("Неверный номер кабинета")

        # TODO: если start_hour >= end_hour -> TimeRangeError
        if start_hour >= end_hour:
            raise TimeRangeError ("Ошибка временного диапазона")

        # TODO: вернуть объект Booking(...)
        return Booking(start_hour, end_hour, booking_id, room_id, owner)

    def submit(self, row: str):
        # TODO: внутри try вызвать parse_booking(row)
        try:
            booking = self.parse_booking(row)

        # TODO: затем schedules[booking.room_id].add_booking(booking)
            self.shedules[booking.room_id].add_booking(booking)
        
        # TODO: успех добавить в self.accepted
            self.accepted.append(booking)

        # TODO: BookingError сохранить в self.errors как (row, error_type, message)
        except BookingError as e:
            self.errors.append((row, type(e).__name__, str(e)))

    def load(self, rows: list[str]):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)

    def busiest_room(self) -> Optional[tuple[str, int]]:
        busiest = None
        max_hours = -1
        # TODO: найти комнату с максимумом total_reserved_hours()
        for room_id, shedule in self.shedules.items():
            hours = shedule.total_reserved_hours()
            if hours > max_hours:
                max_hours = hours
                busiest = (room_id, hours)
        return busiest
        # TODO: вернуть tuple(room_id, total_hours)
    

    def find_booking(self, booking_id: str) -> Optional[Booking]:
        # TODO: вернуть Optional[Booking]
        # TODO: пройтись по всем schedules и по всем bookings внутри них
        # TODO: если booking.booking_id совпал -> вернуть booking
        # TODO: если не найдено -> вернуть None
        pass


service = BookingService(rooms)

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые бронирования
# TODO: вывести ошибки
# TODO: вывести расписание по всем комнатам
# TODO: вывести busiest_room()
# TODO: вывести find_booking('BK-102')